# CNN

extract local features while drastically reducing the number of trainable parameters

convolution in math
the area of two function's overlap 

一阶导数	[1, -1] (Sobel / Prewitt)	发现边缘	计算相邻像素的差值，捕捉变化。
二阶导数	Laplacian 核 (中心为 -4)	发现细节/噪点	计算差值的差值，捕捉变化率的变化。
积分	均值核 (所有元素为 1/N)	模糊 / 降噪	离散累加，将能量平摊到周围区域。


当两个核之间无非线性连接时，可满足交换律并可以合并成一个核，减少计算量 合并核的大小未k1+k2-1

大概为2*cin*cout*[(h+2ph-kh)/sh+1]**2的计算量

In [2]:
import torch
from torch import nn

net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.Sigmoid(),
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10))

X = torch.rand(size=(1, 1, 28, 28), dtype=torch.float32)
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape: \t',X.shape)

Conv2d output shape: 	 torch.Size([1, 6, 28, 28])
Sigmoid output shape: 	 torch.Size([1, 6, 28, 28])
AvgPool2d output shape: 	 torch.Size([1, 6, 14, 14])
Conv2d output shape: 	 torch.Size([1, 16, 10, 10])
Sigmoid output shape: 	 torch.Size([1, 16, 10, 10])
AvgPool2d output shape: 	 torch.Size([1, 16, 5, 5])
Flatten output shape: 	 torch.Size([1, 400])
Linear output shape: 	 torch.Size([1, 120])
Sigmoid output shape: 	 torch.Size([1, 120])
Linear output shape: 	 torch.Size([1, 84])
Sigmoid output shape: 	 torch.Size([1, 84])
Linear output shape: 	 torch.Size([1, 10])


In [ ]:
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=batch_size)

def evaluate_accuracy_gpu(net, data_iter, device):
    if isinstance(net,nn.Module):
        net.eval()
    metric = d2l.Accumulator(2)
    with torch.no_grad():
        for X, y in enumerate(data_iter):
            X = X.to(device)
            y= y.to(device)
            metric.add(d2l.accuracy(net(X),y),X.shape[0])
    return metric[0]/metric[1]


device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
def train(net,train_iter,test_iter,num_epochs,lr,device):
    def init_weights(m):
        if type(m) == nn.Linear or nn.Conv2d:
            nn.init.xavier_normal(m.weight)
    net.apply(init_weights)

    net.to(device)

    optimizer = torch.optim.SGD(net.parameters(),lr=lr)
    loss = nn.CrossEntropyLoss()
    num_batches = len(train_iter)
    for epoch in range(num_epochs):
        metric = d2l.Accumulator(3)
        net.train()
        # 相当于train的模式
        for i,(X,y) in enumerate(train_iter):
            optimizer.zero_grad()
            X, y =X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat,y)
            l.backward()
            optimizer.step()
            
            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.accuracy(y_hat, y), X.shape[0])
            train_l = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
        test_acc = evaluate_accuracy_gpu(net,test_iter)


lr, num_epochs = 0.9, 10
train(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())